# W&B Sweep — Actor-Critic (AC)
Búsqueda de hiperparámetros para el agente AC en el ambiente Simple (CSTR).
- Método: Random Search
- Proyecto W&B: `Tesis_AC_CTRL`
- Arquitectura: simple (solo CTRL)
- Acción: continua

## 1. Instalación e Imports

In [ ]:
import os
import random
import numpy as np
import torch
import wandb
import sys
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
# Clonar desde Github:
!git clone https://github.com/valeriaeskenazi/Control-PID-Adaptativo-Inteligente-mediante-Reinforcement-Learning.git
PROJECT_PATH = '/content/Control-PID-Adaptativo-Inteligente-mediante-Reinforcement-Learning/Version_4'
sys.path.append(PROJECT_PATH)

In [ ]:
from Environment.Simulation_Env.Reactor_CSTR import CSTRSimulator
from Environment.PIDControlEnv_simple import PIDControlEnv_Simple
from Environment.PIDControlEnv_complex import PIDControlEnv_Complex
from Agente.AC.train_AC import ACTrainer
from Agente.AC.algorithm_AC import ACAgent
from Aux.Plots import SimplePlotter, print_summary

print('Imports completados')
print(f'PyTorch: {torch.__version__}')
print(f'Device: {"CUDA" if torch.cuda.is_available() else "CPU"}')

In [ ]:
gpu_info = !nvidia-smi
gpu_info = '\n'.join(gpu_info)
if gpu_info.find('failed') >= 0:
    print('Not connected to a GPU')
else:
    print(gpu_info)

## 2. Login W&B

In [ ]:
!pip install wandb --quiet

In [ ]:
wandb.login()

## 3. Configuración del Sweep

In [ ]:
# ============ LISTAS DE OPCIONES PREDEFINIDAS ============

REWARD_WEIGHTS_OPTIONS = [
    {'error': 1.0, 'tiempo': 0.3,   'overshoot': 0.2,  'energy': 0.1},    # 0: balanceado
    {'error': 2.0, 'tiempo': 0.1,   'overshoot': 0.5,  'energy': 0.1},    # 1: foco en error y overshoot
    {'error': 1.0, 'tiempo': 0.001, 'overshoot': 0.3,  'energy': 0.001},  # 2: config base
    {'error': 3.0, 'tiempo': 0.1,   'overshoot': 0.1,  'energy': 0.05},   # 3: solo error importa
]

HIDDEN_DIMS_OPTIONS = [
    (64, 32),
    (128, 64),
    (128, 128, 64),
    (256, 128, 64),
]

# ============ FIJOS PARA TODOS LOS RUNS ============
SEED                         = 42
N_EPISODES                   = 300
EVAL_FREQUENCY               = 50
EARLY_STOPPING_PATIENCE      = 10
EARLY_STOPPING_MIN_DELTA_PCT = 0.01
N_MANIPULABLE_VARS           = 2
MANIPULABLE_RANGES           = [(300, 420), (99.5, 104)]
DT                           = 1.0
DEVICE                       = 'cuda' if torch.cuda.is_available() else 'cpu'

WANDB_TEAM    = 've326684-universidad-ort-uruguay'
WANDB_PROJECT = 'Tesis_AC_CTRL'

sweep_config = {
    'name':   'ac_cstr_random_search',
    'method': 'random',

    'metric': {
        'name': 'eval_reward',
        'goal': 'maximize'
    },

    'parameters': {

        # ============ AMBIENTE ============
        'max_time_detector':  {'values': [15, 30, 60]},
        'max_steps':          {'values': [20, 50, 100]},
        'reward_dead_band':   {'values': [0.01, 0.02, 0.05]},
        'delta_percent_ctrl': {'values': [0.05, 0.1, 0.2, 0.3]},
        'reward_weights_idx': {'values': [0, 1, 2, 3]},

        # ============ CRITERIOS DE ESTABILIDAD ============
        'error_increase_tolerance': {'values': [1.2, 1.5, 2.0]},
        'max_sign_changes_ratio':   {'values': [0.1, 0.2, 0.3]},
        'max_abrupt_change_ratio':  {'values': [0.03, 0.05, 0.1]},
        'abrupt_change_threshold':  {'values': [0.2, 0.3, 0.5]},

        # ============ AGENTE AC ============
        'hidden_dims_idx': {'values': [0, 1, 2, 3]},
        'lr_actor':        {'values': [0.00001, 0.0001, 0.0003]},
        'lr_critic':       {'values': [0.0001,  0.001,  0.01]},
        'gamma':           {'values': [0.95, 0.99, 0.999]},
        'entropy_coef':    {'values': [0.001, 0.01, 0.05]},
        'batch_size':      {'values': [32, 64, 128]},
        'buffer_size':     {'values': [10000, 50000, 100000]},
        'warmup_steps':    {'values': [200, 500, 1000]},
    }
}

sweep_id = wandb.sweep(sweep_config, project=WANDB_PROJECT, entity=WANDB_TEAM)
print(f'Sweep creado: {sweep_id}')

## 4. Función del Sweep

In [ ]:
def sweep_run():
    # -------- Inicializar run --------
    wandb.init()
    cfg = wandb.config

    # -------- Reproducibilidad --------
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(SEED)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark     = False
    wandb.config.update({'seed': SEED}, allow_val_change=True)

    # -------- Resolver índices --------
    reward_weights = REWARD_WEIGHTS_OPTIONS[cfg.reward_weights_idx]
    hidden_dims    = HIDDEN_DIMS_OPTIONS[cfg.hidden_dims_idx]

    wandb.config.update({
        'reward_weights': str(reward_weights),
        'hidden_dims':    str(hidden_dims),
    }, allow_val_change=True)

    # -------- Configurar CSTR --------
    cstr = CSTRSimulator(
        dt=DT,
        control_limits=(MANIPULABLE_RANGES[0], MANIPULABLE_RANGES[1])
    )

    # -------- Construir config del trainer --------
    trainer_config = {
        # === AMBIENTE ===
        'env_config': {
            'architecture':          'simple',
            'env_type':              'simulation',
            'action_type':           'continuous',
            'n_manipulable_vars':    N_MANIPULABLE_VARS,
            'manipulable_ranges':    MANIPULABLE_RANGES,
            'manipulable_setpoints': None,
            'dt_usuario':            DT,
            'max_steps':             cfg.max_steps,
            'max_time_detector':     cfg.max_time_detector,
            'reward_dead_band':      cfg.reward_dead_band,
            'delta_percent_ctrl':    cfg.delta_percent_ctrl,
            'reward_weights':        reward_weights,
            'pid_limits': [
                (0.01, 50.0),
                (0.001, 1.0),
                (0.0,   1.0)
            ],
            'agent_controller_config': {'agent_type': 'continuous'},
            'env_type_config': {
                'dt': DT,
                'control_limits': (MANIPULABLE_RANGES[0], MANIPULABLE_RANGES[1])
            },
            'stability_config': {
                'error_increase_tolerance': cfg.error_increase_tolerance,
                'max_sign_changes_ratio':   cfg.max_sign_changes_ratio,
                'max_abrupt_change_ratio':  cfg.max_abrupt_change_ratio,
                'abrupt_change_threshold':  cfg.abrupt_change_threshold,
            },
        },

        # === AGENTE AC ===
        'agent_ctrl_config': {
            'algorithm':    'ac',
            'state_dim':    N_MANIPULABLE_VARS * 5,
            'action_dim':   N_MANIPULABLE_VARS * 3,
            'n_vars':       N_MANIPULABLE_VARS,
            'action_type':  'continuous',
            'hidden_dims':  hidden_dims,
            'lr_actor':     cfg.lr_actor,
            'lr_critic':    cfg.lr_critic,
            'gamma':        cfg.gamma,
            'entropy_coef': cfg.entropy_coef,
            'batch_size':   cfg.batch_size,
            'buffer_size':  cfg.buffer_size,
            'warmup_steps': cfg.warmup_steps,
            'device':       DEVICE,
            'seed':         SEED,
        },

        # === ENTRENAMIENTO ===
        'n_episodes':                    N_EPISODES,
        'eval_frequency':                EVAL_FREQUENCY,
        'save_frequency':                9999,
        'log_frequency':                 50,
        'checkpoint_dir':                f'checkpoints/ac_{wandb.run.name}',
        'early_stopping_patience':       EARLY_STOPPING_PATIENCE,
        'early_stopping_min_delta_pct':  EARLY_STOPPING_MIN_DELTA_PCT,
        'use_wandb': True,
    }

    # -------- Conectar CSTR al ambiente --------
    trainer = ACTrainer(trainer_config)
    trainer.env.proceso.connect_external_process(cstr)

    # -------- Entrenar --------
    trainer.train()

    # -------- Métricas finales del run --------
    wandb.log({
        'final_eval_reward':      trainer.best_reward,
        'total_episodes':         len(trainer.episode_rewards),
        'final_reward_mean10':    np.mean(trainer.episode_rewards[-10:]),
        'final_energy_mean10':    np.mean(trainer.episode_energies[-10:]),
        'final_overshoot_mean10': np.mean(trainer.episode_max_overshoots[-10:]),
        'final_actor_loss_mean10':  np.mean(trainer.actor_losses[-10:])  if trainer.actor_losses  else 0,
        'final_critic_loss_mean10': np.mean(trainer.critic_losses[-10:]) if trainer.critic_losses else 0,
    })

    print(f'Run completado: {wandb.run.name}')
    wandb.finish()

## 5. Lanzar Sweep

In [ ]:
wandb.agent(sweep_id, function=sweep_run, count=30)

## 6. Run de Producción (post-sweep)


In [ ]:
# ============ CONFIG RUN PRINCIPAL AC 15K ============

WANDB_ENTITY  = 've326684-universidad-ort-uruguay'
WANDB_PROJECT = 'Tesis_AC_CTRL_PROD'
RUN_NAME      = 'ac_ctrl_15k_prod'

N_EPISODES                   = 15000
EVAL_FREQUENCY               = 100
LOG_FREQUENCY                = 100
SAVE_FREQUENCY               = 2000
EARLY_STOPPING_PATIENCE      = 40
EARLY_STOPPING_MIN_DELTA_PCT = 0.01
SEED                         = 42

trainer_config = {
    'env_config': {
        'architecture'           : 'simple',
        'env_type'               : 'simulation',
        'action_type'            : 'continuous',
        'n_manipulable_vars'     : 2,
        'manipulable_ranges'     : [(300, 420), (99.5, 104)],
        'manipulable_setpoints'  : None,
        'dt_usuario'             : 1.0,
        'max_steps'              : 100,        
        'max_time_detector'      : 60,         
        'reward_dead_band'       : 0.02,       
        'delta_percent_ctrl'     : 0.2,        
        'reward_weights'         : {'error': 1.0, 'tiempo': 0.001, 'overshoot': 0.3, 'energy': 0.001},
        'pid_limits'             : [(0.01, 50.0), (0.001, 1.0), (0.0, 1.0)],
        'agent_controller_config': {'agent_type': 'continuous'},
        'env_type_config'        : {'dt': 1.0, 'control_limits': ((300, 420), (99.5, 104))},
        'stability_config'       : {
            'error_increase_tolerance': 2.0,
            'max_sign_changes_ratio'  : 0.3,
            'max_abrupt_change_ratio' : 0.05,
            'abrupt_change_threshold' : 0.2,
        },
    },
    'agent_ctrl_config': {
        'algorithm'    : 'ac',
        'state_dim'    : 10,
        'action_dim'   : 6,
        'n_vars'       : 2,
        'action_type'  : 'continuous',
        'hidden_dims'  : (64, 32),       
        'lr_actor'     : 1e-05,          
        'lr_critic'    : 1e-03,          
        'gamma'        : 0.95,           
        'entropy_coef' : 0.01,           
        'batch_size'   : 64,             
        'buffer_size'  : 50000,          
        'warmup_steps' : 500,            
        'device'       : 'cuda' if torch.cuda.is_available() else 'cpu',
        'seed'         : SEED,
    },
    'n_episodes'                  : N_EPISODES,
    'eval_frequency'              : EVAL_FREQUENCY,
    'log_frequency'               : LOG_FREQUENCY,
    'save_frequency'              : SAVE_FREQUENCY,
    'checkpoint_dir'              : f'checkpoints/{RUN_NAME}',
    'early_stopping_patience'     : EARLY_STOPPING_PATIENCE,
    'early_stopping_min_delta_pct': EARLY_STOPPING_MIN_DELTA_PCT,
    'use_wandb': True,
}

# ============ REPRODUCIBILIDAD ============
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False

# ============ INIT WANDB ============
wandb.init(
    project = WANDB_PROJECT,
    entity  = WANDB_ENTITY,
    name    = RUN_NAME,
    tags    = ['ac', '15k', 'produccion'],
    config  = trainer_config,
)

# ============ ENTRENAR ============
cstr    = CSTRSimulator(dt=1.0, control_limits=((300, 420), (99.5, 104)))
trainer = ACTrainer(trainer_config)
trainer.env.proceso.connect_external_process(cstr)
trainer.train()

# ============ MÉTRICAS FINALES ============
wandb.log({
    'final_eval_reward'       : trainer.best_reward,
    'total_episodes'          : len(trainer.episode_rewards),
    'final_reward_mean10'     : np.mean(trainer.episode_rewards[-10:]),
    'final_energy_mean10'     : np.mean(trainer.episode_energies[-10:]),
    'final_overshoot_mean10'  : np.mean(trainer.episode_max_overshoots[-10:]),
    'final_actor_loss_mean10' : np.mean(trainer.actor_losses[-10:])  if trainer.actor_losses  else 0,
    'final_critic_loss_mean10': np.mean(trainer.critic_losses[-10:]) if trainer.critic_losses else 0,
}, step=len(trainer.episode_rewards))

wandb.finish()
print(f'Run completado: {RUN_NAME}')

## 7. Preueba de rendimiento

In [ ]:
agent = ACAgent(
    state_dim   = 10,
    action_dim  = 6,        # ← 2 vars × 3 params (Kp, Ki, Kd)
    agent_role  = 'ctrl',
    n_vars      = 2,
    hidden_dims = (64, 32), # ← la del sweep ganador
    lr_actor    = 1e-05,
    lr_critic   = 1e-03,
    gamma       = 0.95,
    entropy_coef   = 0.01,
    batch_size     = 64,
    buffer_size    = 50000,
    warmup_steps   = 500,
    device        = 'cuda' if torch.cuda.is_available() else 'cpu',
    seed          = 42
)

agent.load('/Users/gonzalo/Documents/valeria/MASTER/TESIS/PID_Agent/Version_4/Entrenamiento/CTRL/AC/agent_ctrl_best.pt')
print('Agente cargado')

In [ ]:
cstr = CSTRSimulator(dt=1.0, control_limits=((300, 420), (99.5, 104)))  
cstr.reset()

pid_controllers = [
    PIDController(kp=1.0, ki=0.1, kd=0.01, dt=1.0, output_limits=(300, 420)),
    PIDController(kp=1.0, ki=0.1, kd=0.01, dt=1.0, output_limits=(99.5, 104))  
]
apply_action = ApplyAction(
    delta_percent_ctrl=0.2,
    pid_limits=[(0.01, 50.0), (0.001, 1.0), (0.0, 1.0)],  
    manipulable_ranges=[
        
    ]         
)

T_sp, V_sp = 340.0, 101.0
sps = [T_sp, V_sp]
pvs = [cstr.T_ss, cstr.V_ss]

error_integral = [0.0, 0.0]
error_prev     = [sps[i] - pvs[i] for i in range(2)]

traj_T, traj_V = [pvs[0]], [pvs[1]]
traj_kp0, traj_kp1 = [], []

detector = ResponseTimeDetector(proceso=cstr, env_type='simulation', dt=1.0, tolerance=0.02)

for step in range(20):
    errors = [sps[i] - pvs[i] for i in range(2)]

    for i in range(2):
        error_integral[i] += errors[i] * 1.0
    error_derivative = [errors[i] - error_prev[i] for i in range(2)]
    error_prev = errors.copy()

    state = np.array([
        pvs[0], sps[0], errors[0], error_integral[0], error_derivative[0],
        pvs[1], sps[1], errors[1], error_integral[1], error_derivative[1],
    ], dtype=np.float32)

    action = agent.select_action(state, training=False)
    print(f'Step {step+1:2d} | action={action}', end='')

    pid_params = apply_action.translate(
        action=action,
        agent_type='ctrl',
        action_type='continuous',                        # ← PPO es continuo
        current_values=[(p.kp, p.ki, p.kd) for p in pid_controllers]
    )
    for i, (kp, ki, kd) in enumerate(pid_params):
        pid_controllers[i].kp = kp
        pid_controllers[i].ki = ki
        pid_controllers[i].kd = kd

    traj_kp0.append(pid_controllers[0].kp)
    traj_kp1.append(pid_controllers[1].kp)

    resultado = detector.estimate(pvs_inicial=pvs, sps=sps, pid_controllers=pid_controllers, max_time=60, reset_pid=False)  

    pvs = resultado['pvs_final']
    traj_T.append(pvs[0])
    traj_V.append(pvs[1])
    print(f' | T={pvs[0]:.2f} | V={pvs[1]:.3f} | kp0={pid_controllers[0].kp:.4f} | kp1={pid_controllers[1].kp:.4f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(traj_T, marker='o', color='steelblue')
axes[0].axhline(T_sp, color='red', linestyle='--', label=f'SP={T_sp}')
axes[0].set_title('Temperatura T')
axes[0].set_xlabel('Step')
axes[0].set_ylabel('T (K)')
axes[0].legend()

axes[1].plot(traj_V, marker='o', color='orange')
axes[1].axhline(V_sp, color='red', linestyle='--', label=f'SP={V_sp}')
axes[1].set_title('Volumen V')
axes[1].set_xlabel('Step')
axes[1].set_ylabel('V (m³)')
axes[1].legend()

plt.suptitle('Evaluación agente AC CTRL — T_sp=340K, V_sp=100m³', fontweight='bold')
plt.tight_layout()
plt.show()

## 8. Pruebas con otros SP

In [ ]:
DEVICE             = 'cuda' if torch.cuda.is_available() else 'cpu'
MANIPULABLE_RANGES = [(300, 420), (99.5, 104)]
AC_CHECKPOINT = '/Users/gonzalo/Documents/valeria/MASTER/TESIS/PID_Agent/Version_4/Entrenamiento/CTRL/AC/agent_ctrl_best.pt'

eval_config = {
    'env_config': {
        'architecture'           : 'simple',
        'env_type'               : 'simulation',
        'action_type'            : 'continuous',
        'n_manipulable_vars'     : 2,
        'manipulable_ranges'     : [(300, 420), (99.5, 104)],
        'manipulable_setpoints'  : None,
        'dt_usuario'             : 1.0,
        'max_steps'              : 100,        
        'max_time_detector'      : 60,         
        'reward_dead_band'       : 0.02,       
        'delta_percent_ctrl'     : 0.2,        
        'reward_weights'         : {'error': 1.0, 'tiempo': 0.001, 'overshoot': 0.3, 'energy': 0.001},
        'pid_limits'             : [(0.01, 50.0), (0.001, 1.0), (0.0, 1.0)],
        'agent_controller_config': {'agent_type': 'continuous'},
        'env_type_config'        : {'dt': 1.0, 'control_limits': ((300, 420), (99.5, 104))},
        'stability_config'       : {
            'error_increase_tolerance': 2.0,
            'max_sign_changes_ratio'  : 0.3,
            'max_abrupt_change_ratio' : 0.05,
            'abrupt_change_threshold' : 0.2,
        },
   },
    'agent_ctrl_config': {
        'algorithm'    : 'ac',
        'state_dim'    : 10,
        'action_dim'   : 6,
        'n_vars'       : 2,
        'action_type'  : 'continuous',
        'hidden_dims'  : (64, 32),       
        'lr_actor'     : 1e-05,          
        'lr_critic'    : 1e-03,          
        'gamma'        : 0.95,           
        'entropy_coef' : 0.01,           
        'batch_size'   : 64,             
        'buffer_size'  : 50000,          
        'warmup_steps' : 500,            
        'device'       : 'cuda' if torch.cuda.is_available() else 'cpu',
        'seed'         : SEED,
    },
    'n_episodes'                  : 1,
    'use_wandb': False,
    'checkpoint_dir': 'checkpoints/eval_tmp',
}
cstr = CSTRSimulator(dt=1.0, control_limits=(MANIPULABLE_RANGES[0], MANIPULABLE_RANGES[1]))
trainer = ACTrainer(eval_config)
trainer.env.proceso.connect_external_process(cstr)
trainer.agent_ctrl.load(AC_CHECKPOINT)
print('Agente AC cargado correctamente')


In [ ]:
# ============ GRILLA DE SETPOINTS ============
T_setpoints = [320, 340, 360, 380, 400]   # K
V_setpoints = [100.0, 101.0, 102.0]       # m³

# ============ FUNCIÓN DE EVALUACIÓN ============
def evaluar_sp(trainer, T_sp, V_sp, max_steps=100):
    state = trainer.env.reset()[0]
    trainer.env.manipulable_setpoints = [T_sp, V_sp]
    trainer.env._update_errors()
    state = trainer.env._get_observation()

    print(f'SP después del reset: {trainer.env.manipulable_setpoints}')

    T_hist, V_hist = [], []
    done = False    
    step = 0

    while not done and step < max_steps:
        action = trainer.agent_ctrl.select_action(state, training=False)
        next_state, reward, terminated, truncated, info = trainer.env.step(action)
        done = terminated or truncated

        T_hist.append(trainer.env.manipulable_pvs[0])
        V_hist.append(trainer.env.manipulable_pvs[1])

        state = next_state
        step += 1

    return np.array(T_hist), np.array(V_hist)

In [ ]:
# ============ EVALUAR Y GRAFICAR ============
fig, axes = plt.subplots(
    len(T_setpoints), 2,
    figsize=(14, 4 * len(T_setpoints))
)

for i, T_sp in enumerate(T_setpoints):
    V_sp = 101.0  # V fijo para comparar el efecto de T

    T_hist, V_hist = evaluar_sp(trainer, T_sp, V_sp)
    steps = np.arange(1, len(T_hist) + 1)

    # --- Temperatura ---
    ax_T = axes[i, 0]
    ax_T.plot(steps, T_hist, color='steelblue', linewidth=2, label='T (PV)')
    ax_T.axhline(T_sp, color='red', linestyle='--', linewidth=1.5, label=f'SP={T_sp}K')
    ax_T.set_title(f'Temperatura — SP={T_sp}K, V_sp={V_sp}m³')
    ax_T.set_xlabel('Step')
    ax_T.set_ylabel('T (K)')
    ax_T.legend()
    ax_T.grid(True, alpha=0.3)

    # --- Volumen ---
    ax_V = axes[i, 1]
    ax_V.plot(steps, V_hist, color='orange', linewidth=2, label='V (PV)')
    ax_V.axhline(V_sp, color='red', linestyle='--', linewidth=1.5, label=f'SP={V_sp}m³')
    ax_V.set_title(f'Volumen — SP={T_sp}K, V_sp={V_sp}m³')
    ax_V.set_xlabel('Step')
    ax_V.set_ylabel('V (m³)')
    ax_V.legend()
    ax_V.grid(True, alpha=0.3)

    T_final = T_hist[-1]
    V_final = V_hist[-1]
    print(f'T_sp={T_sp}K → T_final={T_final:.2f}K (error={abs(T_final-T_sp):.2f}K) | '
          f'V_final={V_final:.3f}m³ (error={abs(V_final-V_sp):.4f}m³)')

plt.suptitle('Evaluación AC CTRL — Distintos Setpoints de T', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('eval_ac_ctrl_T_setpoints.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ============ TAMBIÉN VARIAR V ============
fig, axes = plt.subplots(
    len(V_setpoints), 2,
    figsize=(14, 4 * len(V_setpoints))
)

T_sp_fijo = 340.0  # T fija para comparar el efecto de V

for i, V_sp in enumerate(V_setpoints):
    T_hist, V_hist = evaluar_sp(trainer, T_sp_fijo, V_sp)
    steps = np.arange(1, len(T_hist) + 1)

    ax_T = axes[i, 0]
    ax_T.plot(steps, T_hist, color='steelblue', linewidth=2)
    ax_T.axhline(T_sp_fijo, color='red', linestyle='--', linewidth=1.5, label=f'SP={T_sp_fijo}K')
    ax_T.set_title(f'Temperatura — T_sp={T_sp_fijo}K, V_sp={V_sp}m³')
    ax_T.set_xlabel('Step')
    ax_T.set_ylabel('T (K)')
    ax_T.legend()
    ax_T.grid(True, alpha=0.3)

    ax_V = axes[i, 1]
    ax_V.plot(steps, V_hist, color='orange', linewidth=2)
    ax_V.axhline(V_sp, color='red', linestyle='--', linewidth=1.5, label=f'SP={V_sp}m³')
    ax_V.set_title(f'Volumen — T_sp={T_sp_fijo}K, V_sp={V_sp}m³')
    ax_V.set_xlabel('Step')
    ax_V.set_ylabel('V (m³)')
    ax_V.legend()
    ax_V.grid(True, alpha=0.3)

    T_final = T_hist[-1]
    V_final = V_hist[-1]
    print(f'V_sp={V_sp}m³ → V_final={V_final:.3f}m³ (error={abs(V_final-V_sp):.4f}m³) | '
          f'T_final={T_final:.2f}K (error={abs(T_final-T_sp_fijo):.2f}K)')

plt.suptitle('Evaluación AC CTRL — Distintos Setpoints de V', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('eval_ac_ctrl_V_setpoints.png', dpi=150, bbox_inches='tight')
plt.show()